# Solutions to the exercises of Chapter 3

Sample solutions for [Chapter 3, Writing the two functions](../../notebooks/part1_principle/03_writing_the_two_functions.ipynb).

In [1]:
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp

try:
    import onsager as on
except ImportError:          # running from a clone without `pip install -e .`
    import sys
    sys.path.insert(0, "../..")
    import onsager as on

on.use_style()

## Exercise 1. A drop instead of a plate

Inside the drop the kinematics is that of Section 3.5: $\bar v(r) = -r\dot h/2h$ for
$r < R$. The dissipation function is the same integral, now taken to the edge of the
drop, whose radius follows from $\pi R^2h = V$:

$$ \Phi = \frac{3\pi\eta R^4}{4h^3}\,\dot h^2 = \frac{3\eta V^2}{4\pi h^5}\,\dot h^2 . $$

With $A = Fh$, minimising $\Phi + \dot A$ gives $(3\eta V^2/2\pi h^5)\,\dot h = -F$, and
since $d(h^{-4})/dt = -4h^{-5}\dot h$,

$$ \frac{1}{h^4} = \frac{1}{h_0^4} + \frac{8\pi F t}{3\eta V^2} . $$

At long times $h \propto t^{-1/4}$. The drop thins more slowly than the liquid under
a plate of fixed radius ($t^{-1/2}$), because its area, and with it the friction,
grows as it is squeezed.

In [2]:
h, hdot, r = sp.symbols("h hdot r")
F, V, eta = sp.symbols("F V eta", positive=True)
R2 = V / (sp.pi * h)                                     # R**2
Phi = sp.integrate(sp.Rational(1, 2) * 12 * eta / h * (r * hdot / (2 * h)) ** 2 * 2 * sp.pi * r, (r, 0, sp.sqrt(R2)))
print("Phi =", sp.simplify(Phi))
eq = on.kinetic_equations(F * h, Phi, [h], [hdot])
print("hdot =", eq[hdot])
rhs = on.rate_function(eq, [h], [hdot], {F: 1.0, V: 1.0, eta: 1.0})
tt = np.logspace(-2, 4, 61)
num = solve_ivp(rhs, (0, tt[-1]), [0.3], t_eval=tt, rtol=1e-11, atol=1e-14).y[0]
closed = (0.3**-4 + 8 * np.pi * tt / 3) ** -0.25
print(f"largest relative difference from the closed form: {np.max(np.abs(num / closed - 1)):.1e}")

Phi = 3*V**2*eta*hdot**2/(4*pi*h**5)
hdot = -2*pi*F*h**5/(3*V**2*eta)
largest relative difference from the closed form: 1.8e-11


## Exercise 2. Bending energy

Write $c_2$ for the second derivative and $c_4$ for the fourth. For
$A = \int\tfrac12\kappa\,c_2^{\,2}\,dx$,

$$ \delta A = \int\kappa\,c_2\,\frac{d^2\delta c}{dx^2}\,dx = \int\kappa\,c_4\,\delta c\,dx , $$

after two integrations by parts (the boundary terms vanish for a periodic field, or
if $\delta c$ and its first derivative vanish at the ends). Hence

$$ \frac{\delta A}{\delta c} = \kappa\,\frac{d^4c}{dx^4} . $$

The general rule, for an integrand $a(c, c_1, c_2)$ that depends on the field and
its first two derivatives, is

$$ \frac{\delta A}{\delta c} = \frac{\partial a}{\partial c} - \frac{d}{dx}\frac{\partial a}{\partial c_1} + \frac{d^2}{dx^2}\frac{\partial a}{\partial c_2} . $$

The numerical check follows Section 3.3, with the second derivative replaced by
the standard three-point formula.

In [3]:
kappa = 1e-4
field = lambda x: 0.5 + 0.2 * np.sin(2 * np.pi * x) + 0.1 * np.cos(4 * np.pi * x)
d4field = lambda x: 0.2 * (2 * np.pi) ** 4 * np.sin(2 * np.pi * x) + 0.1 * (4 * np.pi) ** 4 * np.cos(4 * np.pi * x)


def discrete_A(c, dx):
    curvature = (np.roll(c, -1) - 2 * c + np.roll(c, 1)) / dx**2
    return np.sum(0.5 * kappa * curvature**2) * dx


Ns = np.array([25, 50, 100, 200])
errors = []
for N in Ns:
    x, dx = np.arange(N) / N, 1.0 / N
    c = field(x)
    num = np.empty(N)
    for i in range(N):
        up, down = c.copy(), c.copy()
        up[i] += 1e-4; down[i] -= 1e-4
        num[i] = (discrete_A(up, dx) - discrete_A(down, dx)) / (2e-4 * dx)
    errors.append(on.sup_error(num, kappa * d4field(x)))
    print(f"N = {N:4d}:  largest difference from the analytical result = {errors[-1]:.2e}")
print(f"slope against the cell width: {on.loglog_slope(1 / Ns, errors)[0]:.2f}")

N =   25:  largest difference from the analytical result = 1.05e-02
N =   50:  largest difference from the analytical result = 2.67e-03
N =  100:  largest difference from the analytical result = 6.76e-04
N =  200:  largest difference from the analytical result = 1.69e-04
slope against the cell width: 1.99


## Exercise 3. Slip

In a film with a free surface the shear stress vanishes at $z = h$, and the
velocity profile that carries a given flow is a parabola,
$u = C\,(b + z - z^2/2h)$, which satisfies the slip condition
$u(0) = b\,du/dz(0)$. Its mean is $\bar v = C\,(b + h/3)$.

There are now two sources of dissipation: viscous shear in the film, at the rate
$\eta\int_0^h (du/dz)^2dz = \eta C^2h/3$ per unit area, and friction at the wall. The
slip condition is a balance between the viscous stress $\eta\,du/dz$ and a wall
friction $(\eta/b)\,u(0)$, so the wall dissipates $(\eta/b)\,u(0)^2 = \eta b\,C^2$.
Together,

$$ 2\Phi = \eta C^2\Big(\frac h3 + b\Big) = \frac{\eta\,\bar v^2}{b + h/3} = \frac{3\eta}{h + 3b}\,\bar v^2 . $$

The friction coefficient $3\eta/h$ becomes $3\eta/(h + 3b)$. When the slip length is
much larger than the thickness the film moves as a plug and the friction is
$\eta/b$, independent of $h$.

In [4]:
z, h, b, C, eta, vbar = sp.symbols("z h b C eta vbar", positive=True)
u = C * (b + z - z**2 / (2 * h))
assert sp.simplify(u.subs(z, 0) - b * sp.diff(u, z).subs(z, 0)) == 0 and sp.diff(u, z).subs(z, h) == 0
C_of_v = sp.solve(sp.integrate(u, (z, 0, h)) / h - vbar, C)[0]
two_Phi = sp.integrate(eta * sp.diff(u, z) ** 2, (z, 0, h)) + eta / b * u.subs(z, 0) ** 2
print("2 Phi per unit area =", sp.factor(sp.simplify(two_Phi.subs(C, C_of_v))))

2 Phi per unit area = 3*eta*vbar**2/(3*b + h)


## Exercise 4. Three vessels

Heights $h_1, h_2, h_3$; flow rates $Q_1$ (from vessel 1 to 2) and $Q_2$ (from 2 to
3); three constraints,

$$ S\dot h_1 + Q_1 = 0 , \qquad S\dot h_2 - Q_1 + Q_2 = 0 , \qquad S\dot h_3 - Q_2 = 0 , $$

with multipliers $p_1, p_2, p_3$, and $\Phi = \tfrac12R_h(Q_1^2 + Q_2^2)$. The
multipliers come out as $\rho g h_i$, and the heights obey

$$ \frac{d}{dt}\begin{pmatrix} h_1 \\ h_2 \\ h_3 \end{pmatrix}
   = -\frac{\rho g}{R_hS}\begin{pmatrix} 1 & -1 & 0 \\ -1 & 2 & -1 \\ 0 & -1 & 1 \end{pmatrix}
   \begin{pmatrix} h_1 \\ h_2 \\ h_3 \end{pmatrix} . $$

The matrix has eigenvalues 0, 1 and 3. The zero belongs to the uniform level,
which does not relax (volume is conserved). The other two give the relaxation
times $\tau_1 = R_hS/\rho g$ (the outer vessels exchange liquid through the middle
one, whose level stays put) and $\tau_2 = R_hS/3\rho g$ (the middle vessel exchanges
with both neighbours at once).

In [5]:
h1, h2, h3, q1, q2 = sp.symbols("h1 h2 h3 q1 q2")
d1, d2, d3, Q1, Q2, p1, p2, p3 = sp.symbols("hdot1 hdot2 hdot3 Q1 Q2 p1 p2 p3")
S, rhog, Rh = sp.symbols("S rho_g R_h", positive=True)
eq = on.kinetic_equations(rhog * S * (h1**2 + h2**2 + h3**2) / 2, Rh * (Q1**2 + Q2**2) / 2,
                          [h1, h2, h3, q1, q2], [d1, d2, d3, Q1, Q2],
                          constraints=[S * d1 + Q1, S * d2 - Q1 + Q2, S * d3 - Q2], multipliers=[p1, p2, p3])
print("multipliers:", eq[p1], ",", eq[p2], ",", eq[p3])
matrix = sp.Matrix([[sp.diff(eq[d], hh) for hh in (h1, h2, h3)] for d in (d1, d2, d3)])
print("relaxation rates:", sorted((-matrix).eigenvals().keys(), key=str))

multipliers: h1*rho_g , h2*rho_g , h3*rho_g
relaxation rates: [0, 3*rho_g/(R_h*S), rho_g/(R_h*S)]
